ML MODELS

In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import (classification_report, roc_auc_score,
                              average_precision_score, confusion_matrix,
                              precision_recall_curve)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# Load preprocessed data 
train_df = pd.read_csv('train_processed.csv')
test_df  = pd.read_csv('test_processed.csv')
smote_df = pd.read_csv('train_smote.csv')

X_train = train_df.drop(columns=['fraud'])
y_train = train_df['fraud']
X_test  = test_df.drop(columns=['fraud'])
y_test  = test_df['fraud']
X_smote = smote_df.drop(columns=['fraud'])
y_smote = smote_df['fraud']

# Evaluation metric
def evaluate_model(name, model, X_te, y_te, results_store):
    y_pred  = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1] if hasattr(model, 'predict_proba') else model.decision_function(X_te)
    report  = classification_report(y_te, y_pred, output_dict=True)
    auc_pr  = average_precision_score(y_te, y_proba)
    auc_roc = roc_auc_score(y_te, y_proba)
    print(f"\n{'='*50}")
    print(f"Model: {name}")
    print(classification_report(y_te, y_pred))
    print(f"AUC-PR:  {auc_pr:.4f} | AUC-ROC: {auc_roc:.4f}")
    results_store[name] = {
        'precision': report['1']['precision'],
        'recall':    report['1']['recall'],
        'f1':        report['1']['f1-score'],
        'auc_pr':    auc_pr,
        'auc_roc':   auc_roc
    }
    return model

ml_results = {}

In [ ]:
# Logistic Regression
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42, n_jobs=-1)
lr.fit(X_smote, y_smote)
evaluate_model('Logistic Regression', lr, X_test, y_test, ml_results)
joblib.dump(lr, 'model_lr.pkl')

# Random Forest
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                             random_state=42, n_jobs=-1)
rf.fit(X_smote, y_smote)
evaluate_model('Random Forest', rf, X_test, y_test, ml_results)
joblib.dump(rf, 'model_rf.pkl')

# XGBoost
scale_pos = (y_smote == 0).sum() / (y_smote == 1).sum()
xgb = XGBClassifier(scale_pos_weight=scale_pos, n_estimators=300,
                     learning_rate=0.05, max_depth=6,
                     use_label_encoder=False, eval_metric='aucpr',
                     random_state=42, n_jobs=-1, device='cpu')
xgb.fit(X_smote, y_smote)
evaluate_model('XGBoost', xgb, X_test, y_test, ml_results)
joblib.dump(xgb, 'model_xgb.pkl')

# SVM (subset for speed)
svm_idx = np.random.choice(len(X_smote), size=min(50000, len(X_smote)), replace=False)
svm = SVC(class_weight='balanced', kernel='rbf', probability=True,
          random_state=42, cache_size=500)
svm.fit(X_smote.iloc[svm_idx], y_smote.iloc[svm_idx])
evaluate_model('SVM', svm, X_test, y_test, ml_results)
joblib.dump(svm, 'model_svm.pkl')

# Save ML results  as csv
pd.DataFrame(ml_results).T.to_csv('results_ml.csv')
print("\nML results saved.")


Model: Logistic Regression
              precision    recall  f1-score   support

           0       1.00      0.98      0.99    117649
           1       0.38      0.97      0.54      1280

    accuracy                           0.98    118929
   macro avg       0.69      0.97      0.77    118929
weighted avg       0.99      0.98      0.99    118929

AUC-PR:  0.9067 | AUC-ROC: 0.9981

Model: Random Forest
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    117649
           1       0.85      0.86      0.85      1280

    accuracy                           1.00    118929
   macro avg       0.92      0.93      0.93    118929
weighted avg       1.00      1.00      1.00    118929

AUC-PR:  0.9314 | AUC-ROC: 0.9982

Model: XGBoost
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    117649
           1       0.81      0.91      0.86      1280

    accuracy                           1.00    1

DL MODELS

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import average_precision_score, classification_report, roc_auc_score

In [ ]:
# Device — MPS for Apple Silicon, else CPU
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

# Tensors
X_tr_t  = torch.FloatTensor(X_smote.values).to(device)
y_tr_t  = torch.FloatTensor(y_smote.values).to(device)
X_te_t  = torch.FloatTensor(X_test.values).to(device)
y_te_t  = torch.FloatTensor(y_test.values).to(device)

train_ds  = TensorDataset(X_tr_t, y_tr_t)
train_dl  = DataLoader(train_ds, batch_size=1024, shuffle=True)
input_dim = X_smote.shape[1]

# Focal Loss
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        bce  = nn.functional.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt   = torch.exp(-bce)
        loss = self.alpha * (1 - pt) ** self.gamma * bce
        return loss.mean()

# Training utility
def train_dl_model(model, epochs=30, lr=1e-3, use_focal=False):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = FocalLoss() if use_focal else nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([scale_pos]).to(device))
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for xb, yb in train_dl:
            optimizer.zero_grad()
            out  = model(xb).squeeze()
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_dl):.4f}")
    return model

def eval_dl_model(name, model, results_store):
    model.eval()
    with torch.no_grad():
        logits = model(X_te_t).squeeze()
        proba  = torch.sigmoid(logits).cpu().numpy()
    preds   = (proba >= 0.5).astype(int)
    auc_pr  = average_precision_score(y_test, proba)
    auc_roc = roc_auc_score(y_test, proba)
    print(f"\n{'='*50}\nModel: {name}")
    print(classification_report(y_test, preds))
    print(f"AUC-PR: {auc_pr:.4f} | AUC-ROC: {auc_roc:.4f}")
    report = classification_report(y_test, preds, output_dict=True)
    results_store[name] = {
        'precision': report['1']['precision'],
        'recall':    report['1']['recall'],
        'f1':        report['1']['f1-score'],
        'auc_pr':    auc_pr,
        'auc_roc':   auc_roc
    }

dl_results = {}

# 2a. Standard ANN
class ANN(nn.Module):
    def __init__(self, inp):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(inp, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),  nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x)

ann = ANN(input_dim).to(device)
print("\nTraining ANN...")
ann = train_dl_model(ann, epochs=30, use_focal=True)
eval_dl_model('ANN', ann, dl_results)
torch.save(ann.state_dict(), 'model_ann.pt')

# 2b. Attention-Based ANN
class AttentionANN(nn.Module):
    def __init__(self, inp):
        super().__init__()
        self.fc1       = nn.Linear(inp, 256)
        self.bn1       = nn.BatchNorm1d(256)
        self.attn      = nn.Linear(256, 256)  # feature-wise attention
        self.fc2       = nn.Linear(256, 128)
        self.bn2       = nn.BatchNorm1d(128)
        self.fc3       = nn.Linear(128, 1)
        self.dropout   = nn.Dropout(0.3)

    def forward(self, x):
        h    = torch.relu(self.bn1(self.fc1(x)))
        attn = torch.softmax(self.attn(h), dim=1)   # attention weights
        h    = h * attn                              # attended features
        h    = self.dropout(torch.relu(self.bn2(self.fc2(h))))
        return self.fc3(h)

attn_ann = AttentionANN(input_dim).to(device)
print("\nTraining Attention-ANN...")
attn_ann = train_dl_model(attn_ann, epochs=30, use_focal=True)
eval_dl_model('Attention-ANN', attn_ann, dl_results)
torch.save(attn_ann.state_dict(), 'model_attn_ann.pt')

# 2c. Autoencoder (anomaly detection)
class Autoencoder(nn.Module):
    def __init__(self, inp):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(inp, 64), nn.ReLU(),
            nn.Linear(64, 32),  nn.ReLU(),
            nn.Linear(32, 16)
        )
        self.decoder = nn.Sequential(
            nn.Linear(16, 32),  nn.ReLU(),
            nn.Linear(32, 64),  nn.ReLU(),
            nn.Linear(64, inp)
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

# Train autoencoder on legitimate transactions only
X_legit = X_tr_t[y_tr_t == 0]
ae_ds   = DataLoader(TensorDataset(X_legit), batch_size=1024, shuffle=True)
ae      = Autoencoder(input_dim).to(device)
ae_opt  = optim.Adam(ae.parameters(), lr=1e-3)
ae_crit = nn.MSELoss()
print("\nTraining Autoencoder...")
for epoch in range(30):
    ae.train()
    for (xb,) in ae_ds:
        ae_opt.zero_grad()
        loss = ae_crit(ae(xb), xb)
        loss.backward()
        ae_opt.step()
    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1}/30 | Recon Loss: {loss.item():.6f}")

ae.eval()
with torch.no_grad():
    recon = ae(X_te_t)
    recon_error = ((X_te_t - recon) ** 2).mean(dim=1).cpu().numpy()

threshold = np.percentile(recon_error, 98)
ae_preds  = (recon_error > threshold).astype(int)
auc_pr    = average_precision_score(y_test, recon_error)
report    = classification_report(y_test, ae_preds, output_dict=True)
print(f"\nAutoencoder | AUC-PR: {auc_pr:.4f}")
print(classification_report(y_test, ae_preds))
dl_results['Autoencoder'] = {
    'precision': report['1']['precision'],
    'recall':    report['1']['recall'],
    'f1':        report['1']['f1-score'],
    'auc_pr':    auc_pr,
    'auc_roc':   roc_auc_score(y_test, recon_error)
}
torch.save(ae.state_dict(), 'model_ae.pt')
pd.DataFrame(dl_results).T.to_csv('results_dl.csv')
print("\nDL results saved.")

Using device: mps

Training ANN...
  Epoch 10/30 | Loss: 0.0019
  Epoch 20/30 | Loss: 0.0016
  Epoch 30/30 | Loss: 0.0014

Model: ANN
              precision    recall  f1-score   support

           0       1.00      0.99      1.00    117649
           1       0.66      0.95      0.78      1280

    accuracy                           0.99    118929
   macro avg       0.83      0.97      0.89    118929
weighted avg       1.00      0.99      0.99    118929

AUC-PR: 0.9394 | AUC-ROC: 0.9990

Training Attention-ANN...
  Epoch 10/30 | Loss: 0.0017
  Epoch 20/30 | Loss: 0.0014
  Epoch 30/30 | Loss: 0.0013

Model: Attention-ANN
              precision    recall  f1-score   support

           0       1.00      0.99      1.00    117649
           1       0.57      0.93      0.71      1280

    accuracy                           0.99    118929
   macro avg       0.79      0.96      0.85    118929
weighted avg       0.99      0.99      0.99    118929

AUC-PR: 0.9128 | AUC-ROC: 0.9942

Training 

GNN Models

In [ ]:
import torch
from torch_geometric.data import Data, HeteroData
from torch_geometric.nn import SAGEConv, GATConv, to_hetero
from torch_geometric.transforms import NormalizeFeatures
import torch.nn.functional as F

# ── Build Transaction Graph ──────────────────────────────────
# Nodes = transactions; edges = shared customer or merchant
# Load original (un-smoted) processed train+test for graph construction
full_df = pd.concat([train_df, test_df]).reset_index(drop=True)
fraud_labels = full_df['fraud'].values
feat_cols    = [c for c in full_df.columns if c != 'fraud']
node_features = torch.FloatTensor(full_df[feat_cols].values)

# Build edges: connect transactions sharing the same customer
# (Assumes 'customer' column is label-encoded integer)
print("Building graph edges...")
cust_col = 'customer' if 'customer' in full_df.columns else feat_cols[0]
edge_index_list = []
for cid, grp in full_df.groupby(cust_col):
    idxs = grp.index.tolist()
    if len(idxs) > 1:
        for i in range(len(idxs) - 1):
            edge_index_list.append([idxs[i], idxs[i+1]])
            edge_index_list.append([idxs[i+1], idxs[i]])

edge_index = torch.LongTensor(edge_index_list).T
labels     = torch.LongTensor(fraud_labels)

# Temporal mask: first 80% train, last 20% test
n_total    = len(full_df)
split_pt   = int(n_total * 0.8)
train_mask = torch.zeros(n_total, dtype=torch.bool)
test_mask  = torch.zeros(n_total, dtype=torch.bool)
train_mask[:split_pt] = True
test_mask[split_pt:]  = True

graph = Data(x=node_features, edge_index=edge_index,
             y=labels, train_mask=train_mask, test_mask=test_mask)
print(f"Graph: {graph.num_nodes} nodes, {graph.num_edges} edges")

gnn_device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
graph = graph.to(gnn_device)

# GNN Evaluation utility
def eval_gnn(name, model, graph, results_store):
    model.eval()
    with torch.no_grad():
        out   = model(graph.x, graph.edge_index)
        proba = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
    preds   = (proba[graph.test_mask.cpu()] >= 0.5).astype(int)
    y_true  = graph.y[graph.test_mask].cpu().numpy()
    y_score = proba[graph.test_mask.cpu()]
    auc_pr  = average_precision_score(y_true, y_score)
    auc_roc = roc_auc_score(y_true, y_score)
    report  = classification_report(y_true, preds, output_dict=True)
    print(f"\n{'='*50}\nModel: {name}")
    print(classification_report(y_true, preds))
    print(f"AUC-PR: {auc_pr:.4f} | AUC-ROC: {auc_roc:.4f}")
    results_store[name] = {
        'precision': report['1']['precision'],
        'recall':    report['1']['recall'],
        'f1':        report['1']['f1-score'],
        'auc_pr':    auc_pr,
        'auc_roc':   auc_roc
    }

def train_gnn(model, graph, epochs=100, lr=0.005):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    # Class-weighted loss for imbalance
    n_fraud  = graph.y[graph.train_mask].sum().item()
    n_legit  = graph.train_mask.sum().item() - n_fraud
    w        = torch.tensor([1.0, n_legit / n_fraud]).to(gnn_device)
    criterion = nn.CrossEntropyLoss(weight=w)
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        out  = model(graph.x, graph.edge_index)
        loss = criterion(out[graph.train_mask], graph.y[graph.train_mask])
        loss.backward()
        optimizer.step()
        if (epoch + 1) % 25 == 0:
            print(f"  Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}")
    return model

gnn_results = {}
feat_dim = graph.num_node_features

# 3a. GraphSAGE
class GraphSAGE(nn.Module):
    def __init__(self, in_ch, hidden=128, out_ch=2):
        super().__init__()
        self.conv1 = SAGEConv(in_ch, hidden)
        self.conv2 = SAGEConv(hidden, hidden)
        self.conv3 = SAGEConv(hidden, out_ch)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = self.dropout(x)
        x = F.relu(self.conv2(x, edge_index))
        x = self.dropout(x)
        return self.conv3(x, edge_index)

sage = GraphSAGE(feat_dim).to(gnn_device)
print("\nTraining GraphSAGE...")
sage = train_gnn(sage, graph, epochs=100)
eval_gnn('GraphSAGE', sage, graph, gnn_results)
torch.save(sage.state_dict(), 'model_graphsage.pt')

# 3b. Graph Attention Network (GAT)
class GAT(nn.Module):
    def __init__(self, in_ch, hidden=64, out_ch=2, heads=4):
        super().__init__()
        self.conv1 = GATConv(in_ch, hidden, heads=heads, dropout=0.3)
        self.conv2 = GATConv(hidden * heads, hidden, heads=1, dropout=0.3)
        self.fc    = nn.Linear(hidden, out_ch)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x, edge_index):
        x = F.elu(self.conv1(x, edge_index))
        x = self.dropout(x)
        x = F.elu(self.conv2(x, edge_index))
        return self.fc(x)

gat = GAT(feat_dim).to(gnn_device)
print("\nTraining GAT...")
gat = train_gnn(gat, graph, epochs=100)
eval_gnn('GAT', gat, graph, gnn_results)
torch.save(gat.state_dict(), 'model_gat.pt')

# 3c. Heterogeneous Graph Transformer (HGT)
# Build HeteroData: customer nodes, merchant nodes, transaction edges
hetero_data = HeteroData()
raw_df = pd.concat([train_df, test_df]).reset_index(drop=True)

# Node feature matrices
cust_ids   = raw_df['customer'].unique()
merch_ids  = raw_df['merchant'].unique() if 'merchant' in raw_df.columns else raw_df[feat_cols[1]].unique()

cust_map   = {v: i for i, v in enumerate(cust_ids)}
merch_map  = {v: i for i, v in enumerate(merch_ids)}

hetero_data['customer'].x  = torch.zeros(len(cust_ids), 8)   # placeholder features
hetero_data['merchant'].x  = torch.zeros(len(merch_ids), 8)
hetero_data['transaction'].x = node_features
hetero_data['transaction'].y  = labels
hetero_data['transaction'].train_mask = train_mask
hetero_data['transaction'].test_mask  = test_mask

# Edges: customer → transaction
c_idx = raw_df['customer'].map(cust_map).values
t_idx = np.arange(len(raw_df))
hetero_data['customer',  'makes',  'transaction'].edge_index = \
    torch.LongTensor([c_idx, t_idx])
hetero_data['transaction', 'rev_makes', 'customer'].edge_index = \
    torch.LongTensor([t_idx, c_idx])

if 'merchant' in raw_df.columns:
    m_idx = raw_df['merchant'].map(merch_map).values
    hetero_data['merchant', 'receives', 'transaction'].edge_index = \
        torch.LongTensor([m_idx, t_idx])
    hetero_data['transaction', 'rev_receives', 'merchant'].edge_index = \
        torch.LongTensor([t_idx, m_idx])

from torch_geometric.nn import HGTConv, Linear

class HGT(nn.Module):
    def __init__(self, hidden=64, num_heads=2, num_layers=2):
        super().__init__()
        self.lin_dict = nn.ModuleDict({
            ntype: Linear(-1, hidden)
            for ntype in hetero_data.node_types
        })
        self.convs = nn.ModuleList([
            HGTConv(hidden, hidden, hetero_data.metadata(), num_heads)
            for _ in range(num_layers)
        ])
        self.clf = Linear(hidden, 2)

    def forward(self, x_dict, edge_index_dict):
        x_dict = {k: self.lin_dict[k](v) for k, v in x_dict.items()}
        for conv in self.convs:
            x_dict = {k: F.relu(v) for k, v in conv(x_dict, edge_index_dict).items()}
        return self.clf(x_dict['transaction'])
n_fraud = int(graph.y[graph.train_mask].sum().item())
n_legit = int(graph.train_mask.sum().item()) - n_fraud

hetero_data = hetero_data.to(gnn_device)
hgt = HGT().to(gnn_device)
hgt_opt = optim.Adam(hgt.parameters(), lr=0.005, weight_decay=5e-4)
criterion_hgt = nn.CrossEntropyLoss(weight=torch.tensor([1.0, n_legit/n_fraud]).to(gnn_device))

print("\nTraining HGT...")
for epoch in range(100):
    hgt.train()
    hgt_opt.zero_grad()
    out  = hgt(hetero_data.x_dict, hetero_data.edge_index_dict)
    loss = criterion_hgt(out[hetero_data['transaction'].train_mask],
                         hetero_data['transaction'].y[hetero_data['transaction'].train_mask])
    loss.backward()
    hgt_opt.step()
    if (epoch + 1) % 25 == 0:
        print(f"  Epoch {epoch+1}/100 | Loss: {loss.item():.4f}")

hgt.eval()
with torch.no_grad():
    out_hgt   = hgt(hetero_data.x_dict, hetero_data.edge_index_dict)
    proba_hgt = torch.softmax(out_hgt, dim=1)[:, 1].cpu().numpy()
mask_np = hetero_data['transaction'].test_mask.cpu().numpy()
y_true  = hetero_data['transaction'].y[hetero_data['transaction'].test_mask].cpu().numpy()
auc_pr  = average_precision_score(y_true, proba_hgt[mask_np])
preds   = (proba_hgt[mask_np] >= 0.5).astype(int)
report  = classification_report(y_true, preds, output_dict=True)
print(f"\nHGT | AUC-PR: {auc_pr:.4f}")
gnn_results['HGT'] = {
    'precision': report['1']['precision'],
    'recall':    report['1']['recall'],
    'f1':        report['1']['f1-score'],
    'auc_pr':    auc_pr,
    'auc_roc':   roc_auc_score(y_true, proba_hgt[mask_np])
}
torch.save(hgt.state_dict(), 'model_hgt.pt')
pd.DataFrame(gnn_results).T.to_csv('results_gnn.csv')
print("GNN results saved.")

Building graph edges...
Graph: 594643 nodes, 1181062 edges

Training GraphSAGE...
  Epoch 25/100 | Loss: 0.0732
  Epoch 50/100 | Loss: 0.0544
  Epoch 75/100 | Loss: 0.0483
  Epoch 100/100 | Loss: 0.0453

Model: GraphSAGE
              precision    recall  f1-score   support

           0       1.00      0.99      0.99    117649
           1       0.44      0.98      0.61      1280

    accuracy                           0.99    118929
   macro avg       0.72      0.98      0.80    118929
weighted avg       0.99      0.99      0.99    118929

AUC-PR: 0.9470 | AUC-ROC: 0.9990

Training GAT...
  Epoch 25/100 | Loss: 0.1164
  Epoch 50/100 | Loss: 0.0961
  Epoch 75/100 | Loss: 0.0927
  Epoch 100/100 | Loss: 0.0881

Model: GAT
              precision    recall  f1-score   support

           0       1.00      0.98      0.99    117649
           1       0.36      0.99      0.53      1280

    accuracy                           0.98    118929
   macro avg       0.68      0.98      0.76    1189

XAI Models

In [ ]:
import shap
import lime
import lime.lime_tabular
import os
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
warnings.filterwarnings("ignore")
os.makedirs("xai_plots", exist_ok=True)

# STEP 0 — Reload data + models
device     = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
gnn_device = device

train_df = pd.read_csv('train_processed.csv')
test_df  = pd.read_csv('test_processed.csv')
smote_df = pd.read_csv('train_smote.csv')

X_train = train_df.drop(columns=['fraud'])
y_train = train_df['fraud']
X_test  = test_df.drop(columns=['fraud'])
y_test  = test_df['fraud']
X_smote = smote_df.drop(columns=['fraud'])
y_smote = smote_df['fraud']

feat_names = X_test.columns.tolist()
feat_cols  = feat_names
input_dim  = X_test.shape[1]

# Reload ML models
lr  = joblib.load('model_lr.pkl')
rf  = joblib.load('model_rf.pkl')
xgb = joblib.load('model_xgb.pkl')
svm = joblib.load('model_svm.pkl')

# Rebuild DL model classes + load weights
class ANN(nn.Module):
    def __init__(self, inp):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(inp, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),  nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x): return self.net(x)

class AttentionANN(nn.Module):
    def __init__(self, inp):
        super().__init__()
        self.fc1     = nn.Linear(inp, 256)
        self.bn1     = nn.BatchNorm1d(256)
        self.attn    = nn.Linear(256, 256)
        self.fc2     = nn.Linear(256, 128)
        self.bn2     = nn.BatchNorm1d(128)
        self.fc3     = nn.Linear(128, 1)
        self.dropout = nn.Dropout(0.3)
    def forward(self, x):
        h    = torch.relu(self.bn1(self.fc1(x)))
        attn = torch.softmax(self.attn(h), dim=1)
        h    = h * attn
        h    = self.dropout(torch.relu(self.bn2(self.fc2(h))))
        return self.fc3(h)

class Autoencoder(nn.Module):
    def __init__(self, inp):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(inp, 64), nn.ReLU(),
            nn.Linear(64, 32),  nn.ReLU(),
            nn.Linear(32, 16)
        )
        self.decoder = nn.Sequential(
            nn.Linear(16, 32), nn.ReLU(),
            nn.Linear(32, 64), nn.ReLU(),
            nn.Linear(64, inp)
        )
    def forward(self, x): return self.decoder(self.encoder(x))

ann = ANN(input_dim).to(device)
ann.load_state_dict(torch.load('model_ann.pt', map_location=device))
ann.eval()

attn_ann = AttentionANN(input_dim).to(device)
attn_ann.load_state_dict(torch.load('model_attn_ann.pt', map_location=device))
attn_ann.eval()

ae = Autoencoder(input_dim).to(device)
ae.load_state_dict(torch.load('model_ae.pt', map_location=device))
ae.eval()

X_te_t = torch.FloatTensor(X_test.values).to(device)

# Rebuild graph + GNN models
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv, GATConv

class GraphSAGE(nn.Module):
    def __init__(self, in_ch, hidden=128, out_ch=2):
        super().__init__()
        self.conv1   = SAGEConv(in_ch, hidden)
        self.conv2   = SAGEConv(hidden, hidden)
        self.conv3   = SAGEConv(hidden, out_ch)
        self.dropout = nn.Dropout(0.3)
    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index)); x = self.dropout(x)
        x = F.relu(self.conv2(x, edge_index)); x = self.dropout(x)
        return self.conv3(x, edge_index)

class GAT(nn.Module):
    def __init__(self, in_ch, hidden=64, out_ch=2, heads=4):
        super().__init__()
        self.conv1   = GATConv(in_ch, hidden, heads=heads, dropout=0.3)
        self.conv2   = GATConv(hidden * heads, hidden, heads=1, dropout=0.3)
        self.fc      = nn.Linear(hidden, out_ch)
        self.dropout = nn.Dropout(0.3)
    def forward(self, x, edge_index):
        x = F.elu(self.conv1(x, edge_index)); x = self.dropout(x)
        x = F.elu(self.conv2(x, edge_index))
        return self.fc(x)

full_df       = pd.concat([train_df, test_df]).reset_index(drop=True)
node_features = torch.FloatTensor(full_df[feat_cols].values)
cust_col      = 'customer' if 'customer' in full_df.columns else feat_cols[0]
edge_index_list = []
for cid, grp in full_df.groupby(cust_col):
    idxs = grp.index.tolist()
    if len(idxs) > 1:
        for i in range(len(idxs) - 1):
            edge_index_list.append([idxs[i],   idxs[i+1]])
            edge_index_list.append([idxs[i+1], idxs[i]])

edge_index = torch.LongTensor(edge_index_list).T
labels     = torch.LongTensor(full_df['fraud'].values)
n_total    = len(full_df)
split_pt   = int(n_total * 0.8)
train_mask = torch.zeros(n_total, dtype=torch.bool)
test_mask  = torch.zeros(n_total, dtype=torch.bool)
train_mask[:split_pt] = True
test_mask[split_pt:]  = True

graph = Data(x=node_features, edge_index=edge_index,
             y=labels, train_mask=train_mask,
             test_mask=test_mask).to(gnn_device)

sage = GraphSAGE(graph.num_node_features).to(gnn_device)
sage.load_state_dict(torch.load('model_graphsage.pt', map_location=gnn_device))
sage.eval()

gat = GAT(graph.num_node_features).to(gnn_device)
gat.load_state_dict(torch.load('model_gat.pt', map_location=gnn_device))
gat.eval()

print("All data and models reloaded.")

# STEP 1 — Build xai_X window guaranteed to contain fraud
fraud_indices_full = y_test[y_test == 1].index.tolist()
if len(fraud_indices_full) == 0:
    raise ValueError("No fraud cases in y_test — check your data.")

fraud_iloc  = X_test.index.get_loc(fraud_indices_full[0])
start_iloc  = max(0, fraud_iloc - 499)
end_iloc    = start_iloc + 500

xai_X = X_test.iloc[start_iloc:end_iloc].reset_index(drop=True)
xai_y = y_test.iloc[start_iloc:end_iloc].reset_index(drop=True)
fraud_loc = int(xai_y[xai_y == 1].index[0])   # guaranteed to exist

background = torch.FloatTensor(X_smote.values[:200]).to(device)
sample_t   = torch.FloatTensor(xai_X.values[:100]).to(device)
TOP_K = min(20, len(feat_names))

print(f"xai_X: {xai_X.shape} | Fraud in window: {(xai_y==1).sum()} | TOP_K: {TOP_K}")

# ── Helper: collapse any SHAP array shape → 1D per-feature ───
# Replace with this (fixed):
def shap_mean_importance(sv):
    arr = np.array(sv)
    if arr.ndim == 3:
        # Multi-class (e.g. RF): shape (samples, features, 2) → take class-1
        # Single-output (e.g. ANN): shape (samples, features, 1) → squeeze last axis
        arr = arr[:, :, -1] if arr.shape[2] > 1 else arr[:, :, 0]
    return np.abs(arr).mean(axis=0)   # → (features,)

# ── Helper: save SHAP bar + beeswarm ─────────────────────────
def save_shap_plots(shap_vals, data, title_prefix, prefix):
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_vals, data, feature_names=feat_names,
                      show=False, plot_type="bar")
    plt.title(f"{title_prefix} — Feature Importance (Bar)")
    plt.tight_layout()
    plt.savefig(f"xai_plots/{prefix}_bar.png", dpi=150, bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_vals, data, feature_names=feat_names, show=False)
    plt.title(f"{title_prefix} — Beeswarm")
    plt.tight_layout()
    plt.savefig(f"xai_plots/{prefix}_beeswarm.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  {title_prefix} SHAP saved.")

# 4A — SHAP: XGBoost (TreeExplainer)

print("\n[1/9] SHAP — XGBoost")
exp_xgb = shap.TreeExplainer(xgb)
sv_xgb  = exp_xgb.shap_values(xai_X)
save_shap_plots(sv_xgb, xai_X, "XGBoost", "01_xgb")

shap_exp = shap.Explanation(
    values=sv_xgb[fraud_loc],
    base_values=exp_xgb.expected_value,
    data=xai_X.iloc[fraud_loc].values,
    feature_names=feat_names
)
plt.figure(figsize=(10, 6))
shap.waterfall_plot(shap_exp, show=False)
plt.tight_layout()
plt.savefig("xai_plots/01_xgb_waterfall.png", dpi=150, bbox_inches="tight")
plt.close()

# 4B — SHAP: Random Forest (TreeExplainer)
print("[2/9] SHAP — Random Forest")
exp_rf = shap.TreeExplainer(rf)
sv_rf  = exp_rf.shap_values(xai_X)
sv_rf1 = sv_rf[1] if isinstance(sv_rf, list) else sv_rf
save_shap_plots(sv_rf1, xai_X, "Random Forest", "02_rf")

# 4C — SHAP: ANN (DeepExplainer)
print("[3/9] SHAP — ANN (DeepExplainer)")
exp_ann = shap.DeepExplainer(ann, background)
sv_ann  = exp_ann.shap_values(sample_t)
sv_ann_ = sv_ann[0] if isinstance(sv_ann, list) else sv_ann
save_shap_plots(sv_ann_, xai_X.iloc[:100], "ANN", "03_ann")

# 4D — SHAP: Attention-ANN (GradientExplainer)
# DeepExplainer fails on softmax attention — GradientExplainer handles it

print("[4/9] SHAP — Attention-ANN (GradientExplainer)")
exp_attn = shap.GradientExplainer(attn_ann, background)
sv_attn  = exp_attn.shap_values(sample_t)
sv_attn_ = sv_attn[0] if isinstance(sv_attn, list) else sv_attn
save_shap_plots(sv_attn_, xai_X.iloc[:100], "Attention-ANN", "04_attn_ann")

# 4E — LIME: Logistic Regression + SVM

print("[5/9] LIME — Logistic Regression & SVM")
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_smote.values,
    feature_names=feat_names,
    class_names=["Legit", "Fraud"],
    mode="classification",
    random_state=42
)
for label_name, label_val in [("fraud", 1), ("legit", 0)]:
    pos = int(xai_y[xai_y == label_val].index[0])
    exp = lime_explainer.explain_instance(
        xai_X.values[pos], lr.predict_proba, num_features=TOP_K)
    fig = exp.as_pyplot_figure()
    fig.suptitle(f"LIME — LR ({label_name.capitalize()})")
    fig.savefig(f"xai_plots/05_lime_lr_{label_name}.png", dpi=150, bbox_inches="tight")
    plt.close()

svm_lime = lime_explainer.explain_instance(
    xai_X.values[fraud_loc], svm.predict_proba, num_features=TOP_K)
fig = svm_lime.as_pyplot_figure()
fig.suptitle("LIME — SVM (Fraud)")
fig.savefig("xai_plots/05_lime_svm_fraud.png", dpi=150, bbox_inches="tight")
plt.close()
print("  LIME plots saved.")

# 4F — Attention Weight Heatmap + Input Feature Importance

print("[6/9] Attention Weight Visualisation")
sample_10 = torch.FloatTensor(xai_X.values[:10]).to(device)
with torch.no_grad():
    h_vis        = torch.relu(attn_ann.bn1(attn_ann.fc1(sample_10)))
    attn_weights = torch.softmax(attn_ann.attn(h_vis), dim=1).cpu().numpy()
# attn_weights: (10, 256) — hidden neurons, NOT input features
TOP_NEURONS  = min(20, attn_weights.shape[1])
top20_idx    = np.argsort(attn_weights.mean(axis=0))[::-1][:TOP_NEURONS]
top20_labels = [f"N{i}" for i in top20_idx]

plt.figure(figsize=(14, 5))
sns.heatmap(attn_weights[:, top20_idx], annot=True, fmt=".3f", cmap="YlOrRd",
            linewidths=0.4,
            xticklabels=top20_labels,
            yticklabels=[f"Tx {i}" for i in range(10)])
plt.title(f"Attention Weights — Top {TOP_NEURONS} Hidden Neurons (10 Transactions)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("xai_plots/06_attention_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()

# Input-level importance via gradient × attention
attn_ann.eval()
sample_10_grad = torch.FloatTensor(xai_X.values[:10]).to(device).requires_grad_(True)
h_g   = torch.relu(attn_ann.bn1(attn_ann.fc1(sample_10_grad)))
aw_g  = torch.softmax(attn_ann.attn(h_g), dim=1)
score = (h_g * aw_g).sum()
score.backward()
input_importance = sample_10_grad.grad.abs().mean(dim=0).cpu().numpy()  # (n_features,)

top_feat_idx   = np.argsort(input_importance)[::-1][:TOP_K]
top_feat_names = [feat_names[i] for i in top_feat_idx]

plt.figure(figsize=(12, 4))
plt.bar(range(TOP_K), input_importance[top_feat_idx], color="tomato", edgecolor="black")
plt.xticks(range(TOP_K), top_feat_names, rotation=45, ha="right")
plt.title(f"Attention-ANN — Input Feature Importance via Gradient × Attention (Top {TOP_K})")
plt.ylabel("Mean |Gradient| × Attention Score")
plt.tight_layout()
plt.savefig("xai_plots/06b_attention_input_features.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Attention heatmap + input-feature importance saved.")

# 4G — Autoencoder: Per-Feature Reconstruction Error

print("[7/9] Autoencoder — Reconstruction Error")
with torch.no_grad():
    recon_full  = ae(X_te_t)
    feat_errors = ((X_te_t - recon_full) ** 2).cpu().numpy()

f_idx  = np.where(y_test.values == 1)[0][:5]
l_idx  = np.where(y_test.values == 0)[0][:5]
avg_f  = feat_errors[f_idx].mean(axis=0)
avg_l  = feat_errors[l_idx].mean(axis=0)
top_ae = min(15, len(feat_names))
top15  = np.argsort(avg_f)[::-1][:top_ae]
x_pos, w = np.arange(top_ae), 0.35

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x_pos - w/2, avg_f[top15], w, label="Fraud",  color="tomato",    edgecolor="black")
ax.bar(x_pos + w/2, avg_l[top15], w, label="Legit",  color="steelblue", edgecolor="black")
ax.set_xticks(x_pos)
ax.set_xticklabels([feat_names[i] for i in top15], rotation=45, ha="right")
ax.set_title(f"Autoencoder — Reconstruction Error: Fraud vs Legit (Top {top_ae})")
ax.set_ylabel("Mean Squared Error")
ax.legend()
plt.tight_layout()
plt.savefig("xai_plots/07_autoencoder_recon_error.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Autoencoder plot saved.")

# 4H — GNNExplainer: GraphSAGE

print("[8/9] GNNExplainer — GraphSAGE")
from torch_geometric.explain import Explainer, GNNExplainer as PygGNNExplainer

explainer_gnn = Explainer(
    model=sage,
    algorithm=PygGNNExplainer(epochs=200),
    explanation_type="model",
    node_mask_type="attributes",
    edge_mask_type="object",
    model_config=dict(mode="multiclass_classification",
                      task_level="node", return_type="raw")
)
fraud_node_idx = int((graph.y[graph.test_mask] == 1).nonzero()[0].item())
explanation    = explainer_gnn(graph.x, graph.edge_index, index=fraud_node_idx)
node_imp       = explanation.node_mask.mean(dim=0).cpu().numpy()
top_gnn        = min(15, len(feat_cols))
top15_gnn      = np.argsort(node_imp)[::-1][:top_gnn]

plt.figure(figsize=(11, 5))
plt.bar(range(top_gnn), node_imp[top15_gnn], color="steelblue", edgecolor="black")
plt.xticks(range(top_gnn), [feat_cols[i] for i in top15_gnn], rotation=45, ha="right")
plt.title(f"GNNExplainer — GraphSAGE: Top {top_gnn} Node Feature Importances")
plt.ylabel("Importance Score")
plt.tight_layout()
plt.savefig("xai_plots/08_gnnexplainer_graphsage.png", dpi=150, bbox_inches="tight")
plt.close()
print("  GNNExplainer saved.")

# 4I — GAT Built-in Attention Weight Distribution

print("[9/9] GAT Attention Weight Distribution")

class GATWithAttn(nn.Module):
    def __init__(self, in_ch, hidden=64, out_ch=2, heads=4):
        super().__init__()
        self.conv1   = GATConv(in_ch, hidden, heads=heads, dropout=0.3)
        self.conv2   = GATConv(hidden * heads, hidden, heads=1, dropout=0.3)
        self.fc      = nn.Linear(hidden, out_ch)
        self.dropout = nn.Dropout(0.3)
    def forward(self, x, edge_index, return_attention=False):
        x, (_, aw1) = self.conv1(x, edge_index, return_attention_weights=True)
        x = F.elu(x); x = self.dropout(x)
        x, (_, aw2) = self.conv2(x, edge_index, return_attention_weights=True)
        x = F.elu(x)
        if return_attention:
            return self.fc(x), aw1, aw2
        return self.fc(x)

gat_xai = GATWithAttn(graph.num_node_features).to(gnn_device)
gat_xai.load_state_dict(gat.state_dict())
gat_xai.eval()
with torch.no_grad():
    _, aw1, _ = gat_xai(graph.x, graph.edge_index, return_attention=True)

aw1_np = aw1.cpu().numpy().flatten()
plt.figure(figsize=(9, 4))
plt.hist(aw1_np, bins=80, color="steelblue", edgecolor="none", alpha=0.85)
plt.axvline(aw1_np.mean(), color="tomato", linestyle="--",
            label=f"Mean = {aw1_np.mean():.4f}")
plt.title("GAT — Attention Weight Distribution (Layer 1, All Heads)")
plt.xlabel("Attention Weight")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.savefig("xai_plots/09_gat_attention_distribution.png", dpi=150, bbox_inches="tight")
plt.close()
print("  GAT attention plot saved.")

# 4J — Cross-Model SHAP Comparison (FIXED: handles 3D RF arrays)

shap_imp = {
    "XGBoost":       shap_mean_importance(sv_xgb),
    "Random Forest": shap_mean_importance(sv_rf1),
    "ANN":           shap_mean_importance(sv_ann_),
    "Attention-ANN": shap_mean_importance(sv_attn_),
}
top_cross   = min(10, len(feat_names))
top10_idx   = np.argsort(shap_imp["XGBoost"])[::-1][:top_cross]
top10_names = [feat_names[i] for i in top10_idx]

fig, ax = plt.subplots(figsize=(13, 6))
x, bar_w = np.arange(top_cross), 0.2
colors   = ["steelblue", "tomato", "seagreen", "darkorange"]
for i, (name, imp) in enumerate(shap_imp.items()):
    ax.bar(x + i * bar_w, imp[top10_idx], bar_w,
           label=name, color=colors[i], edgecolor="black", linewidth=0.5)
ax.set_xticks(x + bar_w * 1.5)
ax.set_xticklabels(top10_names, rotation=45, ha="right")
ax.set_title(f"SHAP Cross-Model Feature Importance Comparison (Top {top_cross})")
ax.set_ylabel("Mean |SHAP Value|")
ax.legend()
plt.tight_layout()
plt.savefig("xai_plots/10_shap_cross_model.png", dpi=150, bbox_inches="tight")
plt.close()

# Final Summary

plots = sorted(os.listdir("xai_plots"))
print(f"\n=== XAI COMPLETE — {len(plots)} plots saved to xai_plots/ ===")
for p in plots:
    print(f"  {p}")

All data and models reloaded.
xai_X: (500, 14) | Fraud in window: 1 | TOP_K: 14

[1/9] SHAP — XGBoost
  XGBoost SHAP saved.
[2/9] SHAP — Random Forest
  Random Forest SHAP saved.
[3/9] SHAP — ANN (DeepExplainer)
  ANN SHAP saved.
[4/9] SHAP — Attention-ANN (GradientExplainer)
  Attention-ANN SHAP saved.
[5/9] LIME — Logistic Regression & SVM
  LIME plots saved.
[6/9] Attention Weight Visualisation
  Attention heatmap + input-feature importance saved.
[7/9] Autoencoder — Reconstruction Error
  Autoencoder plot saved.
[8/9] GNNExplainer — GraphSAGE
  GNNExplainer saved.
[9/9] GAT Attention Weight Distribution
  GAT attention plot saved.

=== XAI COMPLETE — 18 plots saved to xai_plots/ ===
  01_xgb_bar.png
  01_xgb_beeswarm.png
  01_xgb_waterfall.png
  02_rf_bar.png
  02_rf_beeswarm.png
  03_ann_bar.png
  03_ann_beeswarm.png
  04_attn_ann_bar.png
  04_attn_ann_beeswarm.png
  05_lime_lr_fraud.png
  05_lime_lr_legit.png
  05_lime_svm_fraud.png
  06_attention_heatmap.png
  06b_attention_inpu